In [ ]:
# 📌 Install and Import Libraries
!pip install selenium pandas beautifulsoup4 --quiet

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import StaleElementReferenceException
from bs4 import BeautifulSoup
import pandas as pd
import time

In [ ]:
import os
import pandas as pd

checkpoint_file = "csc_jobs_checkpoint.csv"
start_page = 1
all_jobs = []

if os.path.exists(checkpoint_file):
    checkpoint_df = pd.read_csv(checkpoint_file)
    all_jobs = checkpoint_df.to_dict("records")
    start_page = (len(all_jobs) // 100) + 1  # Assuming 100 entries per page
    print(f"🔁 Resuming from page {start_page} with {len(all_jobs)} entries.")
else:
    print("🆕 Starting fresh.")

In [ ]:
# 3. Launch Browser and Load Page
# Make sure chromedriver is in your PATH or provide the full path below
driver = webdriver.Chrome()  # or webdriver.Chrome(executable_path="path/to/chromedriver")
driver.get("https://csc.gov.ph/career/index.php")

# Optional: maximize window for full content load
driver.maximize_window()

In [ ]:
# Optional: Determine name of the table
tables = driver.find_elements(By.TAG_NAME, "table")
print(f"Found {len(tables)} tables on the page.")

for i, table in enumerate(tables):
    print(f"\n🔎 Table {i+1}")
    print("ID:", table.get_attribute("id"))
    print("Class:", table.get_attribute("class"))
    print("Text preview:", table.text[:200])

In [ ]:
# Set entries per page to 100
Select(WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.NAME, "jobs_length"))
)).select_by_visible_text("100")
time.sleep(2)

In [ ]:
start_page = 1
page = start_page
all_jobs = []

while True:
    print(f"📄 Scraping page {page}...")

    try:
        # Parse current page
        soup = BeautifulSoup(driver.page_source, "html.parser")
        table = soup.find("table", {"id": "jobs"})
        rows = table.find("tbody").find_all("tr")

        for row in rows:
            cols = row.find_all("td")
            if len(cols) >= 6:
                job = {
                    "Agency": cols[0].text.strip(),
                    "Region": cols[1].text.strip(),
                    "Position Title": cols[2].text.strip(),
                    "Plantilla Item No.": cols[3].text.strip(),
                    "Posting Date": cols[4].text.strip(),
                    "Closing Date": cols[5].text.strip()
                }
                all_jobs.append(job)

        # Get current page number
        current_page = driver.find_element(By.CSS_SELECTOR, "#jobs_paginate .paginate_button.current").text.strip()

        # Click "Next"
        next_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.ID, "jobs_next"))
        )

        if "disabled" in next_button.get_attribute("class"):
            print("✅ Reached last page.")
            break

        next_button.click()

        # Wait for page number to change
        WebDriverWait(driver, 10).until(
            lambda d: d.find_element(By.CSS_SELECTOR, "#jobs_paginate .paginate_button.current").text.strip() != current_page
        )

        page += 1
        time.sleep(1)

    except Exception as e:
        print(f"❌ Error navigating page {page}: {e}")

        # Save partial results
        end_page = page
        filename = f"csc_all_job_listings_({start_page}-{end_page}).csv"
        pd.DataFrame(all_jobs).to_csv(filename, index=False)
        print(f"💾 Saved partial results to {filename} with {len(all_jobs)} entries.")

        # Reset for next batch
        start_page = page + 1
        page = start_page
        all_jobs = []

        # Skip to next page
        try:
            current_page = int(driver.find_element(By.CSS_SELECTOR, "#jobs_paginate .paginate_button.current").text.strip())
            while current_page < start_page:
                print(f"⏩ Skipping page {current_page}...")
                next_button = WebDriverWait(driver, 10).until(
                    EC.element_to_be_clickable((By.ID, "jobs_next"))
                )
                next_button.click()
                WebDriverWait(driver, 10).until(
                    lambda d: int(d.find_element(By.CSS_SELECTOR, "#jobs_paginate .paginate_button.current").text.strip()) > current_page
                )
                current_page = int(driver.find_element(By.CSS_SELECTOR, "#jobs_paginate .paginate_button.current").text.strip())
                time.sleep(1)
        except Exception as skip_error:
            print(f"⚠️ Failed to skip to page {start_page}: {skip_error}")
            break

In [ ]:
# 📌 Save Results to CSV
from datetime import date

today = date.today()
df = pd.DataFrame(all_jobs)
df.to_csv(f"csc_{len(all_jobs)}_listings_{today}.csv", index=False)
print(f"✅ Scraped {len(all_jobs)} job entries across {page} pages.")

In [ ]:
# 📌 Clean Up
driver.quit()